<a href="https://colab.research.google.com/github/neha-lokireddy/BDA-Assignment/blob/main/bda_assignmnet_q2__083.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## K-Means Clustering with Apache Spark

This notebook demonstrates how to build a K-Means clustering model using Apache Spark MLlib. We will use the Iris dataset, a common dataset for classification and clustering tasks, to illustrate the process.

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
# Initialize SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KMeansClustering") \
    .getOrCreate()

print("Spark Session Initialized.")

### 1. Load the Dataset

We'll load the Iris dataset from a public URL. The dataset contains measurements for 150 iris flowers, with 4 features (sepal length, sepal width, petal length, petal width) and their corresponding species.

In [ ]:
# Load the Iris dataset
# Using a public CSV URL for demonstration
iris_data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"

# Define schema since Spark's inferSchema might get types wrong for generic CSVs without headers
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

schema = StructType([
    StructField("sepal_length", DoubleType(), True),
    StructField("sepal_width", DoubleType(), True),
    StructField("petal_length", DoubleType(), True),
    StructField("petal_width", DoubleType(), True),
    StructField("species", StringType(), True)
])

df = spark.read.csv(iris_data_url, schema=schema)

print("Dataset loaded successfully.")
df.printSchema()
df.show(5)

### 2. Data Preprocessing

For K-Means clustering, we need to assemble all feature columns into a single vector column. It's also good practice to scale the features to prevent features with larger values from dominating the distance calculations.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Define feature columns (excluding 'species' for clustering)
feature_columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

# Assemble features into a single vector column
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
df_assembled = assembler.transform(df)

# Scale the features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print("Features assembled and scaled.")
df_scaled.show(5)

### 3. K-Means Clustering

Now, we will apply the K-Means algorithm. For the Iris dataset, we know there are 3 species, so we will set the number of clusters (k) to 3. K-Means attempts to partition observations into `k` clusters in which each observation belongs to the cluster with the nearest mean (cluster centers or cluster centroid), serving as a prototype of the cluster.

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Set number of clusters (k)
k = 3

# Create a KMeans instance
kmeans = KMeans(featuresCol="scaledFeatures", k=k, seed=1)

# Train the K-Means model
kmeans_model = kmeans.fit(df_scaled)

# Make predictions (assign each data point to a cluster)
predictions = kmeans_model.transform(df_scaled)

print("K-Means model trained and predictions made.")
predictions.show(5)

### 4. Model Evaluation

We can evaluate the clustering using the Silhouette Score. The Silhouette Score is a measure of how similar an object is to its own cluster (cohesion) compared to other clusters (separation). A higher Silhouette Score indicates better-defined clusters.

In [ ]:
# Evaluate clustering by computing Silhouette Score
evaluator = ClusteringEvaluator(featuresCol="scaledFeatures", predictionCol="prediction", metricName="silhouette")
silhouette = evaluator.evaluate(predictions)

print(f"Silhouette Score for K-Means with k={k}: {silhouette}")

# Show the cluster centers
centers = kmeans_model.clusterCenters()
print("Cluster Centers:")
for center in centers:
    print(center)

### 5. Visualize Results (Optional)

To better understand the clusters, we can convert the Spark DataFrame to a Pandas DataFrame and visualize them. Since the Iris dataset has 4 features, we'll use a pairplot to visualize the clusters across different feature pairs. We'll also include the original species for comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert Spark DataFrame to Pandas DataFrame for visualization
pandas_df = predictions.toPandas()

# Map numerical predictions to a more readable format if desired (e.g., Cluster 0, Cluster 1)
pandas_df['predicted_cluster'] = 'Cluster ' + pandas_df['prediction'].astype(str)

# Create a pairplot to visualize the clusters
# We'll use two features for simplicity in the pairplot for demonstration purposes,
# but you can plot all feature pairs.
# sns.pairplot(pandas_df, vars=feature_columns, hue='predicted_cluster', palette='viridis')

# Let's visualize with two features and color by predicted cluster
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='petal_length',
    y='petal_width',
    hue='predicted_cluster',
    data=pandas_df,
    palette='viridis',
    s=100,
    alpha=0.8
)
plt.title('K-Means Clusters of Iris Dataset (Petal Length vs Petal Width)')
plt.xlabel('Petal Length')
plt.ylabel('Petal Width')
plt.legend(title='Predicted Cluster')
plt.grid(True)
plt.show()

# For comparison, let's also see the original species distribution
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='petal_length',
    y='petal_width',
    hue='species',
    data=pandas_df,
    palette='tab10',
    s=100,
    alpha=0.8
)
plt.title('Original Species of Iris Dataset (Petal Length vs Petal Width)')
plt.xlabel('Petal Length')
plt.ylabel('Petal Width')
plt.legend(title='Species')
plt.grid(True)
plt.show()

# Stop SparkSession
spark.stop()
print("Spark Session stopped.")